In [1]:
import sys
print(sys.executable)

D:\pytorch_project\pytorch_env\Scripts\python.exe


##  Imports & Environment Setup

In [2]:
from groq import Groq
from dotenv import load_dotenv
import os
import re
import json
import random
from datetime import datetime

load_dotenv()

client = Groq(api_key=os.getenv("GROQ_API_KEY"))

## Validation Functions

In [3]:
def validate_aadhaar(value):
    value = value.strip().replace(" ", "")
    if not value.isdigit():
        return False, "Aadhaar must contain only numbers"
    if len(value) != 12:
        return False, f"Aadhaar must be exactly 12 digits, you entered {len(value)}"
    return True, "Valid"

def validate_pan(value):
    value = value.strip().upper()
    pattern = r'^[A-Z]{5}[0-9]{4}[A-Z]{1}$'
    if not re.match(pattern, value):
        return False, "PAN must be in format ABCDE1234F (5 letters, 4 numbers, 1 letter)"
    return True, "Valid"

def validate_mobile(value):
    value = value.strip().replace(" ", "")
    if not value.isdigit():
        return False, "Mobile number must contain only digits"
    if len(value) != 10:
        return False, "Mobile number must be exactly 10 digits"
    if value[0] not in ['6', '7', '8', '9']:
        return False, "Mobile number must start with 6, 7, 8, or 9"
    return True, "Valid"

def validate_email(value):
    value = value.strip()
    pattern = r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$'
    if not re.match(pattern, value):
        return False, "Please enter a valid email address like example@gmail.com"
    return True, "Valid"

def validate_dob(value):
    value = value.strip()
    formats = ["%d/%m/%Y", "%d-%m-%Y", "%Y-%m-%d", "%d %m %Y"]
    for fmt in formats:
        try:
            dob = datetime.strptime(value, fmt)
            age = (datetime.now() - dob).days // 365
            if age < 18:
                return False, f"You must be at least 18 years old. Your age appears to be {age}"
            if age > 100:
                return False, "Please enter a valid date of birth"
            return True, "Valid"
        except ValueError:
            continue
    return False, "Please enter date in DD/MM/YYYY format"

## Application State

In [4]:
conversation_history = []
current_intent = "form_filling"  # tracks active intent

application_data = {
    # Personal Details
    "full_name": None,
    "aadhaar_number": None,
    "pan_number": None,
    "date_of_birth": None,
    "gender": None,
    "category": None,
    "mobile_number": None,
    "email": None,
    "residential_address": None,
    # Business Details
    "business_name": None,
    "business_type": None,
    "business_description": None,
    "business_address": None,
    "business_status": None,
    "years_in_operation": None,
    "number_of_employees": None,
    # Loan Details
    "loan_category": None,
    "loan_amount": None,
    "loan_purpose": None,
    "preferred_bank": None,
    # Documents
    "documents_provided": []
}

## System Prompt

In [5]:
# ── FORM FILLING PROMPT (unchanged) ──────────────────────────────────────────
form_filling_prompt = """
ROLE
You are a Government Digital Assistant helping citizens apply for a PM Mudra Yojana loan.
Your job is to collect information step-by-step and complete a structured application form.

IMPORTANT PRINCIPLE
This is a structured form collection task, not a general conversation.
You must only ask for missing information and validate user input.

The current application state will be provided separately as:
APPLICATION_DATA (fields already collected and fields still missing)

Your task is to determine the next field needed and ask the user for it.


APPLICATION FORM STRUCTURE

SECTION 1 — PERSONAL DETAILS
1. full_name
2. aadhaar_number (12 digits)
3. pan_number (ABCDE1234F format)
4. date_of_birth (DD/MM/YYYY)
5. gender (Male/Female/Other)
6. category (General/SC/ST/OBC/Minority)
7. mobile_number (10 digits)
8. email
9. residential_address

SECTION 2 — BUSINESS DETAILS
10. business_name
11. business_type (Manufacturing/Trading/Services)
12. business_description
13. business_address
14. business_status (Existing/New)
15. years_in_operation (ONLY if Existing)
16. employee_count

SECTION 3 — LOAN DETAILS
17. loan_category (Shishu/Kishore/Tarun)
18. loan_amount
19. loan_purpose (equipment / working capital / expansion)
20. preferred_bank

SECTION 4 — DOCUMENT CONFIRMATION
21. confirm_documents_ready

DOCUMENTS REQUIRED

Mandatory:
- Aadhaar card
- PAN card
- Passport size photo
- Address proof of business
- Bank statement (last 6 months)

Conditional:
- Business proof (if business is existing)
- Caste certificate (if category is SC/ST/OBC/Minority)


STRICT VALIDATION RULES

AADHAAR: Must be exactly 12 digits. No spaces, letters, or symbols.
PAN: Format: 5 uppercase letters + 4 digits + 1 uppercase letter. Example: ABCDE1234F
MOBILE: Must be exactly 10 digits. Must start with 6, 7, 8, or 9.

NEVER guess missing numbers, correct the user's value, or reject valid values.


CONVERSATION RULES

1. Ask ONE question at a time.
2. Always ask for the next missing field in the form.
3. If the user provides multiple answers in one message, extract all valid ones.
4. If business_status = New, skip years_in_operation.
5. If category = General, caste certificate not required.
6. If business address = residential address, reuse it.

Always acknowledge the user's answer briefly before asking the next question.
Use clear and simple language.
Detect the language the user is writing in and always respond in that same language.


APPLICATION COMPLETION

When all fields are collected:
1. Show a structured summary of the application.
2. Ask the user to review the details.
3. Ask: "Do you want to SUBMIT the application or change anything?"

Only when the user explicitly says "Submit", output:
FORM_COMPLETE
<JSON application object>

Do NOT output FORM_COMPLETE unless the user explicitly confirms submission.
"""


# ── SCHEME INFO PROMPT ────────────────────────────────────────────────────────
scheme_info_prompt = """
ROLE
You are a knowledgeable Government Assistant specializing in the PM Mudra Yojana scheme.
Answer the user's questions clearly and accurately about the scheme.

TOPICS YOU CAN COVER
- What is PM Mudra Yojana
- Eligibility criteria
- Loan categories: Shishu (up to ₹50,000), Kishore (₹50,001–₹5 lakh), Tarun (₹5 lakh–₹10 lakh)
- Purpose of the loan
- Required documents
- How to apply
- Interest rates and repayment
- Which banks and NBFCs offer Mudra loans
- Government subsidies or benefits

RULES
- Only answer questions related to PM Mudra Yojana and its application process.
- Be concise, friendly, and easy to understand.
- If the user asks something outside this scope, politely redirect them.
- Detect the language the user is writing in and respond in the same language.
- At the end of your answer, add a gentle nudge:
  "Would you like to continue with your application or do you have more questions?"
"""


# ── STATUS PROMPT ─────────────────────────────────────────────────────────────
status_prompt = """
ROLE
You are a Government Assistant helping users check the status of their PM Mudra Yojana application.

RULES
- The current application data will be provided to you.
- Summarize what fields have been filled and what fields are still pending.
- If the application is complete, confirm it and show the Application ID if available.
- Be concise and friendly.
- Detect the language the user is writing in and respond in the same language.
- At the end, ask: "Would you like to continue filling the form or do you have any questions?"
"""


# ── OFF-TOPIC PROMPT ──────────────────────────────────────────────────────────
offtopic_prompt = """
ROLE
You are a focused Government Assistant for PM Mudra Yojana loan applications.

RULES
- The user has asked something unrelated to PM Mudra Yojana or their application.
- Politely let them know you can only help with:
  1. Filling the Mudra loan application form
  2. Answering questions about the PM Mudra Yojana scheme
  3. Checking their current application status
- Do NOT answer the off-topic question.
- Keep the response short, friendly, and redirect them.
- Detect the language the user is writing in and respond in the same language.
"""

## RAG Imports

In [6]:
import chromadb
from chromadb.config import Settings
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
import numpy as np
import requests

In [7]:
OLLAMA_URL = "http://localhost:11434/api/generate"
OLLAMA_MODEL = "llama3"

def call_ollama(prompt):
    """Helper to call local Ollama model and return response text"""
    try:
        response = requests.post(
            OLLAMA_URL,
            json={
                "model": OLLAMA_MODEL,
                "prompt": prompt,
                "stream": False
            },
            timeout=60
        )
        response.raise_for_status()
        return response.json().get("response", "").strip()
    except Exception as e:
        print(f"⚠️ Ollama call failed: {e}")
        return None

##  Initialize Embedding Model & ChromaDB

In [8]:
# Load embedding model locally
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# ChromaDB for scheme documents (RAG)
doc_chroma_client = chromadb.PersistentClient(path="./mudra_rag_db")
collection = doc_chroma_client.get_or_create_collection(
    name="mudra_scheme_docs",
    metadata={"heuristic": "cosine"}
)

# ChromaDB for conversation history (separate instance)
history_chroma_client = chromadb.PersistentClient(path="./mudra_history_db")
history_collection = history_chroma_client.get_or_create_collection(
    name="conversation_history",
    metadata={"heuristic": "cosine"}
)

print("✅ Embedding model and ChromaDB instances initialized.")

✅ Embedding model and ChromaDB instances initialized.


## Conversation History Store & Retrieval

In [9]:
history_chunks = []
history_bm25 = None
history_message_counter = 0

# Rolling window of recent messages for context generation
recent_messages_window = []
CONTEXT_WINDOW_SIZE = 3  # last 3 messages used as "document" for context


def generate_message_context(recent_messages, current_message):
    """
    Use Ollama to generate a short context that situates the current
    message within the recent conversation window.
    Generated on the fly as each message arrives.
    """
    if not recent_messages:
        return ""

    # Format recent messages as a mini "document"
    conversation_window = "\n".join(recent_messages)

    prompt = f"""<document>
{conversation_window}
</document>

Here is the message we want to situate within the conversation:
<chunk>
{current_message}
</chunk>

Please give a short succinct context to situate this message within the
overall conversation for the purposes of improving search retrieval of
the message. Answer only with the succinct context and nothing else."""

    context = call_ollama(prompt)

    if context:
        return context.strip()

    return ""


def store_message(role, content):
    """
    Store a message into ChromaDB history and BM25 with contextual augmentation.
    Context is generated on the fly using the last 3 messages as window.
    """
    global history_chunks, history_bm25, history_message_counter, recent_messages_window

    # Format raw message with role
    raw_message = f"[{role.upper()}]: {content}"

    # Generate context using recent messages window (on the fly)
    context = generate_message_context(recent_messages_window, raw_message)

    # Augment message with context (prepend — Anthropic style)
    if context:
        augmented_message = f"{context}\n\n{raw_message}"
    else:
        augmented_message = raw_message

    # Store augmented message in ChromaDB
    embedding = embedding_model.encode([augmented_message]).tolist()
    history_collection.add(
        documents=[augmented_message],
        embeddings=embedding,
        ids=[f"msg_{history_message_counter}"],
        metadatas=[{
            "role": role,
            "index": history_message_counter,
            "raw_message": raw_message
        }]
    )

    # Store augmented message in memory for BM25
    history_chunks.append(augmented_message)

    # Rebuild BM25 index
    tokenized = [chunk.lower().split() for chunk in history_chunks]
    history_bm25 = BM25Okapi(tokenized)

    # Update rolling window with raw message (not augmented)
    recent_messages_window.append(raw_message)
    if len(recent_messages_window) > CONTEXT_WINDOW_SIZE:
        recent_messages_window.pop(0)

    history_message_counter += 1


def retrieve_relevant_history(current_query, top_n=3):
    """
    Retrieve top_n most relevant past messages using dual retrieval
    (vector + BM25) over contextually augmented conversation history.
    """
    if len(history_chunks) <= 1:
        return None

    # ── Vector retrieval ──────────────────────────────────────────────────────
    query_embedding = embedding_model.encode([current_query]).tolist()
    n_results = min(top_n, len(history_chunks) - 1)
    vector_results = history_collection.query(
        query_embeddings=query_embedding,
        n_results=n_results
    )
    # Return raw messages from metadata for cleaner LLM context
    vector_chunks = [
        r["raw_message"]
        for r in vector_results["metadatas"][0]
        if "raw_message" in r
    ]

    # ── BM25 retrieval ────────────────────────────────────────────────────────
    bm25_chunks = []
    if history_bm25 and len(history_chunks) > 1:
        tokenized_query = current_query.lower().split()
        scores = history_bm25.get_scores(tokenized_query)
        scores[-1] = -1  # exclude last message
        top_indices = np.argsort(scores)[::-1][:top_n]
        # Return raw messages from metadata
        all_metadata = history_collection.get()["metadatas"]
        bm25_chunks = [
            all_metadata[i]["raw_message"]
            for i in top_indices
            if scores[i] > 0 and i < len(all_metadata)
        ]

    # ── Merge & deduplicate ───────────────────────────────────────────────────
    seen = set()
    merged = []
    for chunk in vector_chunks + bm25_chunks:
        if chunk not in seen:
            seen.add(chunk)
            merged.append(chunk)

    if not merged:
        return None

    return "\n".join(merged[:top_n])


def reset_history():
    """Clear all conversation history from ChromaDB and memory"""
    global history_chunks, history_bm25, history_message_counter, recent_messages_window

    all_ids = history_collection.get()["ids"]
    if all_ids:
        history_collection.delete(ids=all_ids)

    history_chunks = []
    history_bm25 = None
    history_message_counter = 0
    recent_messages_window = []
    print("✅ Conversation history cleared.")


## Document Ingestion & Chunking (Paragraph-level)

In [10]:
from langchain_community.document_loaders import PyPDFLoader, TextLoader

# ── ADD YOUR DOCUMENTS HERE ───────────────────────────────────────────────────



document_files = [
    r"D:\Downloads\PMMY_Reference_Document.pdf",

]

# ─────────────────────────────────────────────────────────────────────────────


def load_file(file_path):
    """
    Load a PDF or TXT file and return a list of page/document strings.
    Each page of a PDF becomes one document.
    """
    file_path = file_path.strip()

    if file_path.endswith(".pdf"):
        print(f"📂 Loading PDF: {file_path}")
        loader = PyPDFLoader(file_path)
        pages = loader.load()
        docs = [page.page_content for page in pages if page.page_content.strip()]
        print(f"   ✅ Loaded {len(docs)} pages.")
        return docs

    elif file_path.endswith(".txt"):
        print(f"📂 Loading TXT: {file_path}")
        loader = TextLoader(file_path, encoding="utf-8")
        docs = loader.load()
        texts = [doc.page_content for doc in docs if doc.page_content.strip()]
        print(f"   ✅ Loaded {len(texts)} text blocks.")
        return texts

    else:
        print(f"⚠️ Unsupported file type: {file_path}. Skipping.")
        return []


def chunk_by_paragraph(text, min_chunk_words=30, max_chunk_words=200):
    """
    Smarter chunking strategy:
    - Splits on double newlines (real paragraph breaks)
    - Merges short lines (bullet points, table rows) with the preceding chunk
    - Splits oversized chunks at sentence boundaries
    - Filters out page headers, footers, and single-line noise
    """
    # Step 1: Normalize line endings
    text = text.replace('\r\n', '\n').replace('\r', '\n')

    # Step 2: Split on double newlines (true paragraph breaks)
    raw_paragraphs = [p.strip() for p in text.split('\n\n') if p.strip()]

    # Step 3: If no double newlines found (flat text), split on single newlines
    # and group bullets/short lines together
    if len(raw_paragraphs) <= 2:
        lines = [l.strip() for l in text.split('\n') if l.strip()]
        raw_paragraphs = []
        current = []
        for line in lines:
            word_count = len(line.split())
            # Short line = bullet, table row, header — group with current block
            if word_count < 15:
                current.append(line)
            else:
                # Long line = new paragraph
                if current:
                    raw_paragraphs.append(' '.join(current))
                    current = []
                current.append(line)
        if current:
            raw_paragraphs.append(' '.join(current))

    # Step 4: Filter out noise (page headers, footers, version lines)
    noise_patterns = [
        "comprehensive reference document",
        "source: official",
        "version 1.0",
        "page ",
        "rbi guidelines"
    ]
    filtered = []
    for p in raw_paragraphs:
        if any(noise.lower() in p.lower() for noise in noise_patterns):
            continue
        if len(p.split()) < 5:  # skip very short fragments
            continue
        filtered.append(p)

    # Step 5: Merge undersized chunks with the next chunk
    merged = []
    buffer = ""
    for para in filtered:
        word_count = len((buffer + " " + para).split())
        if len(buffer.split()) < min_chunk_words:
            buffer = (buffer + " " + para).strip()
        elif word_count <= max_chunk_words:
            buffer = (buffer + " " + para).strip()
        else:
            if buffer:
                merged.append(buffer)
            buffer = para

    if buffer:
        merged.append(buffer)

    # Step 6: Split oversized chunks at sentence boundaries
    final_chunks = []
    for chunk in merged:
        if len(chunk.split()) <= max_chunk_words:
            final_chunks.append(chunk)
        else:
            # Split at sentence boundaries
            sentences = re.split(r'(?<=[.!?])\s+', chunk)
            current = ""
            for sentence in sentences:
                if len((current + " " + sentence).split()) <= max_chunk_words:
                    current = (current + " " + sentence).strip()
                else:
                    if current:
                        final_chunks.append(current)
                    current = sentence
            if current:
                final_chunks.append(current)

    return final_chunks


def generate_chunk_context(whole_document, chunk):
    """
    Use Ollama to generate a short 50-100 token context that situates
    the chunk within the whole document. Anthropic-style contextual retrieval.
    """
    prompt = f"""<document>
{whole_document}
</document>

Here is the chunk we want to situate within the whole document:
<chunk>
{chunk}
</chunk>

Please give a short succinct context to situate this chunk within the
overall document for the purposes of improving search retrieval of the
chunk. Answer only with the succinct context and nothing else."""

    context = call_ollama(prompt)

    if context:
        return context.strip()
    return ""


def augment_chunk(context, chunk):
    """Prepend generated context to chunk — Anthropic style"""
    if context:
        return f"{context}\n\n{chunk}"
    return chunk


def ingest_documents(file_paths):
    """
    Load files, chunk by paragraph, generate contextual summaries,
    augment, embed and store into ChromaDB.
    """
    # Load all files into one list of documents
    all_documents = []
    for fp in file_paths:
        loaded = load_file(fp)
        all_documents.extend(loaded)

    if not all_documents:
        print("⚠️ No documents loaded. Please check your file paths.")
        return []

    print(f"\n📚 Total documents to process: {len(all_documents)}")

    all_augmented_chunks = []
    all_raw_chunks = []

    for doc_idx, document in enumerate(all_documents):
        chunks = chunk_by_paragraph(document)
        print(f"\n📄 Processing document {doc_idx + 1}/{len(all_documents)} "
              f"— {len(chunks)} chunks...")

        for chunk_idx, chunk in enumerate(chunks):
            print(f"   Generating context for chunk {chunk_idx + 1}/{len(chunks)}...")

            # Generate context using Ollama
            context = generate_chunk_context(document, chunk)

            # Augment chunk with context (prepend)
            augmented = augment_chunk(context, chunk)

            all_augmented_chunks.append(augmented)
            all_raw_chunks.append(chunk)

    # Clear existing chunks before re-ingesting
    existing_ids = collection.get()["ids"]
    if existing_ids:
        collection.delete(ids=existing_ids)
        print(f"\n🗑️ Cleared {len(existing_ids)} existing chunks.")

    # Embed augmented chunks
    print("\n🔢 Embedding augmented chunks...")
    embeddings = embedding_model.encode(all_augmented_chunks).tolist()

    # Store augmented chunks in ChromaDB
    collection.add(
        documents=all_augmented_chunks,
        embeddings=embeddings,
        ids=[f"chunk_{i}" for i in range(len(all_augmented_chunks))],
        metadatas=[{"raw_chunk": all_raw_chunks[i]} for i in range(len(all_raw_chunks))]
    )

    print(f"\n✅ Ingested {len(all_augmented_chunks)} contextually augmented chunks.")
    return all_raw_chunks


# Run ingestion
all_chunks = ingest_documents(document_files)

📂 Loading PDF: D:\Downloads\PMMY_Reference_Document.pdf
   ✅ Loaded 10 pages.

📚 Total documents to process: 10

📄 Processing document 1/10 — 0 chunks...

📄 Processing document 2/10 — 3 chunks...
   Generating context for chunk 1/3...
   Generating context for chunk 2/3...
   Generating context for chunk 3/3...

📄 Processing document 3/10 — 2 chunks...
   Generating context for chunk 1/2...
   Generating context for chunk 2/2...

📄 Processing document 4/10 — 3 chunks...
   Generating context for chunk 1/3...
   Generating context for chunk 2/3...
   Generating context for chunk 3/3...

📄 Processing document 5/10 — 2 chunks...
   Generating context for chunk 1/2...
   Generating context for chunk 2/2...

📄 Processing document 6/10 — 3 chunks...
   Generating context for chunk 1/3...
   Generating context for chunk 2/3...
   Generating context for chunk 3/3...

📄 Processing document 7/10 — 2 chunks...
   Generating context for chunk 1/2...
   Generating context for chunk 2/2...

📄 Proces

## BM25 Index Setup

In [11]:
def build_bm25_index(chunks):
    """Build BM25 index from chunks"""
    tokenized = [chunk.lower().split() for chunk in chunks]
    bm25 = BM25Okapi(tokenized)
    return bm25

bm25_index = build_bm25_index(all_chunks)
print(f"✅ BM25 index built over {len(all_chunks)} chunks.")

✅ BM25 index built over 23 chunks.


## Query Rewriting

In [12]:

def rewrite_query(user_query):
    """
    Use local Ollama model to generate 3 different search queries
    from the user's question.
    Returns a list of 3 query strings.
    """
    prompt = f"""You are a search query rewriter for a PM Mudra Yojana information retrieval system.

Given the user's question, generate exactly 3 different search queries that together cover
different angles of the question. These will be used to retrieve relevant document chunks.

Rules:
- Each query should be distinct and capture a different aspect
- Keep queries concise (under 15 words each)
- Output ONLY a JSON array of 3 strings, nothing else
- No explanation, no preamble, no markdown

Example output: ["query one", "query two", "query three"]

Original question: {user_query}"""

    raw = call_ollama(prompt)

    if raw:
        try:
            # Strip any accidental markdown backticks
            clean = raw.replace("```json", "").replace("```", "").strip()
            queries = json.loads(clean)
            if isinstance(queries, list) and len(queries) == 3:
                return queries
        except Exception:
            pass

    print("⚠️ Query rewriting failed, using original query.")
    return [user_query, user_query, user_query]

## Dual Retrieval (Vector + BM25)

In [13]:
def vector_search(query, top_k=5):
    """Retrieve top_k chunks using vector similarity from ChromaDB"""
    query_embedding = embedding_model.encode([query]).tolist()
    results = collection.query(
        query_embeddings=query_embedding,
        n_results=top_k
    )
    return results["documents"][0]  # list of chunk strings


def bm25_search(query, top_k=5):
    """Retrieve top_k chunks using BM25 keyword search"""
    tokenized_query = query.lower().split()
    scores = bm25_index.get_scores(tokenized_query)
    top_indices = np.argsort(scores)[::-1][:top_k]
    return [all_chunks[i] for i in top_indices]


def dual_retrieval(queries, top_k=5):
    """
    Run both vector and BM25 search for each query.
    Merge all results and deduplicate.
    """
    seen = set()
    merged = []

    for query in queries:
        vector_results = vector_search(query, top_k=top_k)
        bm25_results   = bm25_search(query, top_k=top_k)

        for chunk in vector_results + bm25_results:
            if chunk not in seen:
                seen.add(chunk)
                merged.append(chunk)

    return merged

##  LLM-based Reranker

In [14]:
def rerank_chunks(user_query, chunks, top_n=5):
    """
    Use local Ollama model to rerank retrieved chunks by relevance
    to the user query. Returns top_n most relevant chunks.
    """
    if not chunks:
        return []

    numbered_chunks = "\n\n".join(
        [f"[{i}] {chunk}" for i, chunk in enumerate(chunks)]
    )

    prompt = f"""You are a document relevance ranker for a PM Mudra Yojana assistant.

Given the user's question and a list of document chunks, return the indices of the
{top_n} most relevant chunks in order from most to least relevant.

Rules:
- Output ONLY a JSON array of {top_n} integers (indices)
- No explanation, no preamble, no markdown backticks
- Example output: [2, 0, 4, 1, 3]

User Question: {user_query}

Document Chunks:
{numbered_chunks}"""

    raw = call_ollama(prompt)

    if raw:
        try:
            clean = raw.replace("```json", "").replace("```", "").strip()
            indices = json.loads(clean)
            valid_indices = [i for i in indices if 0 <= i < len(chunks)][:top_n]
            if valid_indices:
                return [chunks[i] for i in valid_indices]
        except Exception:
            pass

    print("⚠️ Reranking failed, returning top chunks as-is.")
    return chunks[:top_n]

## RAG Pipeline Function

In [15]:
def retrieve_context(user_query, top_n=5):
    """
    Full RAG pipeline:
    1. Rewrite query into 3 variants
    2. Dual retrieval (vector + BM25) for each variant
    3. Rerank merged results
    4. Return top_n chunks as a single context string
    """
    print(f"   [RAG] Rewriting query...")
    queries = rewrite_query(user_query)
    print(f"   [RAG] Queries: {queries}")

    print(f"   [RAG] Running dual retrieval...")
    candidate_chunks = dual_retrieval(queries, top_k=5)
    print(f"   [RAG] Retrieved {len(candidate_chunks)} unique candidate chunks.")

    print(f"   [RAG] Reranking...")
    top_chunks = rerank_chunks(user_query, candidate_chunks, top_n=top_n)
    print(f"   [RAG] Top {len(top_chunks)} chunks selected.")

    # Format as a single context block
    context = "\n\n---\n\n".join(top_chunks)
    return context

## Helper Functions

In [16]:
def get_fields_needed():
    needed = []
    for key, value in application_data.items():
        if value is None:
            needed.append(key)
    if (application_data.get("business_status") or "").lower() == "new":
        if "years_in_operation" in needed:
            needed.remove("years_in_operation")
    return needed

def build_dynamic_prompt():
    collected = {k: v for k, v in application_data.items() if v is not None}
    needed = get_fields_needed()
    return f"""
CURRENT APPLICATION STATE:
Fields collected so far: {json.dumps(collected, indent=2)}
Fields still needed: {json.dumps(needed, indent=2)}
"""

def detect_intent(user_message):
    prompt = f"""You are an intent classifier for a PM Mudra Yojana loan application chatbot.

Classify the user's message into EXACTLY one of these intents:
- form_filling   : User is filling the application form, providing personal/business/loan details, or wants to apply
- scheme_info    : User is asking about what the scheme is, eligibility, loan types, documents, interest rates, banks, benefits
- status         : User is asking about the status of their current application or what they have filled so far
- off_topic      : User is asking something completely unrelated to Mudra Yojana or their application

Reply with ONLY one word — the intent label. No explanation, no punctuation.

User message: {user_message}"""

    raw = call_ollama(prompt)

    if raw:
        intent = raw.strip().lower().split()[0]
        if intent in ["form_filling", "scheme_info", "status", "off_topic"]:
            return intent

    print("⚠️ Intent detection failed, defaulting to form_filling.")
    return "form_filling"

def get_prompt_for_intent(intent, rag_context=None):
    if intent == "scheme_info":
        context_block = f"""
RELEVANT DOCUMENT CONTEXT (use this to answer the user's question):
---
{rag_context if rag_context else "No context retrieved."}
---
"""
        return scheme_info_prompt + context_block

    mapping = {
        "form_filling": form_filling_prompt,
        "status":       status_prompt,
        "off_topic":    offtopic_prompt,
    }
    return mapping.get(intent, form_filling_prompt)

def extract_form_data(response_text):
    if "FORM_COMPLETE" not in response_text:
        return None
    try:
        json_start = response_text.find("{", response_text.find("FORM_COMPLETE"))
        json_end = response_text.rfind("}") + 1
        json_str = response_text[json_start:json_end]
        return json.loads(json_str)
    except Exception as e:
        print(f"Could not parse form data: {e}")
        return None

def update_application_data(extracted_data):
    if not extracted_data:
        return
    for key in application_data:
        if key in extracted_data and extracted_data[key]:
            application_data[key] = extracted_data[key]

def generate_application_id():
    prefix = "MU"
    date_str = datetime.now().strftime("%Y%m%d")
    random_number = random.randint(1000, 9999)
    return f"{prefix}-{date_str}-{random_number}"

def reset():
    global current_intent
    current_intent = "form_filling"

    # Reset application data
    for key in application_data:
        application_data[key] = None
    application_data["documents_provided"] = []

    # Reset conversation history (ChromaDB + memory)
    reset_history()

    print("✅ Application data and conversation history reset.")

## Core Chat Function

In [17]:
def chat(user_message):
    global current_intent

    # Step 1: Detect intent via Ollama
    current_intent = detect_intent(user_message)
    print(f"   [intent: {current_intent}]")

    # Step 2: Retrieve relevant past history via RAG (excluding last message)
    relevant_history = retrieve_relevant_history(user_message, top_n=3)

    # Step 3: Store current user message into history
    store_message("user", user_message)

    # Step 4: Run document RAG only for scheme_info
    rag_context = None
    if current_intent == "scheme_info":
        rag_context = retrieve_context(user_message)

    # Step 5: Pick the right prompt
    system_prompt = get_prompt_for_intent(current_intent, rag_context=rag_context)

    # Step 6: Append dynamic form state for form_filling and status
    if current_intent in ["form_filling", "status"]:
        system_prompt += build_dynamic_prompt()

    # Step 7: Build messages for LLM
    # Only last message is passed directly — rest comes from RAG history
    messages = [{"role": "system", "content": system_prompt}]

    # Inject relevant history as a system context block (not as chat turns)
    if relevant_history:
        messages.append({
            "role": "system",
            "content": f"""RELEVANT CONVERSATION HISTORY (for context only):
---
{relevant_history}
---"""
        })

    # Add only the current user message
    messages.append({
        "role": "user",
        "content": user_message
    })

    # Step 8: Call Groq API
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=messages,
        max_tokens=1000
    )

    assistant_message = response.choices[0].message.content

    # Step 9: Store assistant response into history
    store_message("assistant", assistant_message)

    # Step 10: Handle form completion
    if "FORM_COMPLETE" in assistant_message:
        extracted = extract_form_data(assistant_message)
        update_application_data(extracted)
        clean_response = assistant_message.split("FORM_COMPLETE")[0].strip()
        application_id = generate_application_id()
        print(f"\n🤖 Assistant: {clean_response}")
        print(f"\n✅ Application submitted! Your Application ID: {application_id}")
        print(f"\n📋 Final Application Data:\n{json.dumps(application_data, indent=2)}")
        return

    print(f"\n🤖 Assistant: {assistant_message}")

## Run the Demo (Interactive Loop)


In [18]:
print("🏦 PM Mudra Yojana Loan Application Assistant")
print("Type 'reset' to start over, 'status' to see current data, or 'quit' to exit.\n")

# Start the conversation
chat("Hello, I want to apply for a Mudra loan.")

while True:
    user_input = input("\nYou: ").strip()
    if not user_input:
        continue
    if user_input.lower() == "quit":
        print("Exiting. Goodbye!")
        break
    elif user_input.lower() == "reset":
        reset()
        chat("Hello, I want to apply for a Mudra loan.")
    elif user_input.lower() == "status":
        print(f"\n📋 Current Application Data:\n{json.dumps(application_data, indent=2)}")
    else:
        chat(user_input)

🏦 PM Mudra Yojana Loan Application Assistant
Type 'reset' to start over, 'status' to see current data, or 'quit' to exit.

   [intent: form_filling]

🤖 Assistant: To start your Mudra loan application, I need some information from you. Can you please tell me your full name?



You:  can you tell me what this loan is about


   [intent: scheme_info]
   [RAG] Rewriting query...
   [RAG] Queries: ['What is PM Mudra Yojana', 'Features of PM Mudra Yojana loan', 'Eligibility criteria for PM Mudra Yojana']
   [RAG] Running dual retrieval...
   [RAG] Retrieved 17 unique candidate chunks.
   [RAG] Reranking...
⚠️ Ollama call failed: HTTPConnectionPool(host='localhost', port=11434): Read timed out. (read timeout=60)
⚠️ Reranking failed, returning top chunks as-is.
   [RAG] Top 5 chunks selected.

🤖 Assistant: The Pradhan Mantri MUDRA Yojana (PMMY) is a flagship scheme launched by the Government of India to provide affordable institutional credit to non-corporate, non-farm micro and small enterprises across India. The scheme aims to support individuals who are self-employed, running small businesses, or planning to start a micro enterprise.

Under PMMY, you can avail loans up to ₹10 lakh, categorized into three types:
1. Shishu: up to ₹50,000
2. Kishore: ₹50,001 to ₹5 lakh
3. Tarun: ₹5 lakh to ₹10 lakh

These loans 


You:  quit


Exiting. Goodbye!


In [19]:
def check_ollama():
    """Check if Ollama is running and llama3 is available"""

    # Step 1: Check if Ollama server is reachable
    print("🔍 Checking Ollama server...")
    try:
        response = requests.get("http://localhost:11434/api/tags", timeout=5)
        response.raise_for_status()
        models = [m["name"] for m in response.json().get("models", [])]
        print(f"✅ Ollama server is running.")
        print(f"📦 Available models: {models}")
    except Exception as e:
        print(f"❌ Ollama server not reachable: {e}")
        print("👉 Make sure Ollama is running with: ollama serve")
        return

    # Step 2: Check if llama3 is in the list
    if any("llama3" in m for m in models):
        print("✅ llama3 model is available.")
    else:
        print("❌ llama3 not found. Run: ollama pull llama3")
        return

    # Step 3: Send a real test prompt and print response
    print("\n🔍 Sending test prompt to llama3...")
    test_response = call_ollama("Reply with only this exact text: OLLAMA_IS_WORKING")
    if test_response:
        print(f"✅ Ollama responded: {test_response}")
    else:
        print("❌ Ollama did not respond to test prompt.")

    # Step 4: Test query rewriting specifically
    print("\n🔍 Testing query rewriting with Ollama...")
    queries = rewrite_query("What is a Mudra loan?")
    print(f"✅ Rewritten queries: {queries}")

    # Step 5: Test reranking specifically
    print("\n🔍 Testing reranking with Ollama...")
    dummy_chunks = [
        "Mudra loans are provided to small businesses.",
        "The weather today is sunny.",
        "Shishu category covers loans up to Rs 50,000."
    ]
    reranked = rerank_chunks("What is a Mudra loan?", dummy_chunks, top_n=2)
    print(f"✅ Reranked chunks: {reranked}")

    print("\n🎉 All checks passed! Ollama is working correctly.")

check_ollama()

🔍 Checking Ollama server...
✅ Ollama server is running.
📦 Available models: ['llama3:latest']
✅ llama3 model is available.

🔍 Sending test prompt to llama3...
✅ Ollama responded: OLLAMA_IS_WORKING

🔍 Testing query rewriting with Ollama...
✅ Rewritten queries: ['What is Mudra loan scheme', 'Features of Mudra loan', 'Mudra loan benefits']

🔍 Testing reranking with Ollama...
✅ Reranked chunks: ['Shishu category covers loans up to Rs 50,000.', 'Mudra loans are provided to small businesses.']

🎉 All checks passed! Ollama is working correctly.


In [20]:
# ── Auto inspect after ingestion ──────────────────────────────────────────────

results = collection.get(include=["documents", "metadatas"])
total = len(results["ids"])

print(f"\n{'='*80}")
print(f"📦 CHUNK INSPECTION REPORT")
print(f"{'='*80}")
print(f"Total chunks stored: {total}")

# Word count stats
word_counts = [len(results["documents"][i].split()) for i in range(total)]
raw_word_counts = [len(results["metadatas"][i].get("raw_chunk", "").split()) for i in range(total)]

print(f"\n📊 Augmented Chunk Size Stats (what ChromaDB searches):")
print(f"   Min words : {min(word_counts)}")
print(f"   Max words : {max(word_counts)}")
print(f"   Avg words : {round(sum(word_counts) / total)}")

print(f"\n📊 Raw Chunk Size Stats (what LLM sees):")
print(f"   Min words : {min(raw_word_counts)}")
print(f"   Max words : {max(raw_word_counts)}")
print(f"   Avg words : {round(sum(raw_word_counts) / total)}")

print(f"\n{'='*80}")
print(f"📝 CHUNK PREVIEW (first 5 chunks)")
print(f"{'='*80}")

for i in range(min(5, total)):
    raw = results["metadatas"][i].get("raw_chunk", "N/A")
    augmented = results["documents"][i]
    raw_words = len(raw.split())
    aug_words = len(augmented.split())

    print(f"\n🔷 Chunk {i+1} | Raw: {raw_words} words | Augmented: {aug_words} words")
    print(f"\n   📌 Context (prepended by Ollama):")

    # Extract just the context part (everything before the raw chunk)
    if raw in augmented:
        context_part = augmented.replace(raw, "").strip()
        print(f"   {context_part if context_part else '⚠️ No context generated'}")
    else:
        print(f"   ⚠️ Could not isolate context")

    print(f"\n   📄 Raw Chunk:")
    # Show first 300 characters to keep output readable
    preview = raw[:300] + "..." if len(raw) > 300 else raw
    print(f"   {preview}")
    print(f"\n{'-'*80}")

print(f"\n💡 To inspect more chunks run:")
print(f"   inspect_chunks(show_augmented=True, max_chunks=20)")


📦 CHUNK INSPECTION REPORT
Total chunks stored: 23

📊 Augmented Chunk Size Stats (what ChromaDB searches):
   Min words : 31
   Max words : 229
   Avg words : 152

📊 Raw Chunk Size Stats (what LLM sees):
   Min words : 15
   Max words : 200
   Avg words : 132

📝 CHUNK PREVIEW (first 5 chunks)

🔷 Chunk 1 | Raw: 169 words | Augmented: 187 words

   📌 Context (prepended by Ollama):
   Introduction and overview of the Pradhan Mantri MUDRA Yojana (PMMY) scheme, highlighting its objectives, implementation, and target audience.

   📄 Raw Chunk:
   The Pradhan Mantri MUDRA Yojana (PMMY) is a flagship scheme launched by the Government of India on 8 April 2015. Its primary objective is to provide affordable institutional credit to non-corporate, non-farm micro and small enterprises across India. The scheme is implemented through MUDRA, which sta...

--------------------------------------------------------------------------------

🔷 Chunk 2 | Raw: 160 words | Augmented: 189 words

   📌 Context (p